# TV2 — 02. Tiền xử lý sub-01
Giai đoạn 3: nhìn tín hiệu thô, tìm kênh mắt/Pz/Fz, PSD, kênh nghi hỏng.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne_bids import BIDSPath, read_raw_bids
from eegoddball import config  # cần `pip install -e .`

SUB = "01"

## 3.1 Đọc raw và in `raw.info`

In [ ]:
bids_path = BIDSPath(subject=SUB, task="oddball", datatype="eeg", root=config.DATA_DIR)
raw = read_raw_bids(bids_path, verbose="error")
raw.load_data()
print(raw.info)
print("Số kênh:", len(raw.ch_names), "| sfreq:", raw.info["sfreq"], "Hz | thời lượng:", round(raw.times[-1] / 60, 1), "phút")
print("Loại kênh:", pd.Series(raw.get_channel_types()).value_counts().to_dict())
# kiểm tra đơn vị: EEG tính bằng volt thì độ lệch chuẩn cỡ 1e-5
print("Độ lệch chuẩn trung vị (V):", np.median(raw.get_data().std(axis=1)))

## 3.2 Tên kênh; kênh trán gần mắt, Pz, Fz; gắn montage

In [ ]:
print(raw.ch_names)
for name in ["Fp1", "Fp2", "AFp1", "AFp2", "AF7", "AF8", "Fz", "Pz", "FCz"]:
    print(f"{name:5s}", "có" if name in raw.ch_names else "KHÔNG có")
print("Tên có dấu cách:", [c for c in raw.ch_names if " " in c])

# MNE >= 1.13 đổi tên standard_1005 -> colin27_1005 (cùng tọa độ); giữ tương thích bản cũ
MONTAGE = "colin27_1005" if "colin27_1005" in mne.channels.get_builtin_montages() else "standard_1005"
montage = mne.channels.make_standard_montage(MONTAGE)
print(f"Không có trong {MONTAGE}:", sorted(set(raw.ch_names) - set(montage.ch_names)))

In [ ]:
# đổi tên 2 kênh tai cho khớp montage chuẩn (dataset không có electrodes.tsv)
raw.rename_channels({"TP9 LEFT EAR": "TP9", "TP10 RIGHT EAR": "TP10"})
# O9/O10 là tên khác của I1/I2 trong hệ 10-05 (Oostenveld & Praamstra 2001): mượn tọa độ, giữ tên kênh
pos = montage.get_positions()
pos["ch_pos"]["O9"], pos["ch_pos"]["O10"] = pos["ch_pos"]["I1"], pos["ch_pos"]["I2"]
montage = mne.channels.make_dig_montage(**pos)
raw.set_montage(montage)
print("Kênh chưa có tọa độ:", [c for c, p in zip(raw.ch_names, raw._get_channel_positions()) if np.isnan(p).any()] or "không có")

raw.plot_sensors(show_names=["Fp1", "Fp2", "AFp2", "Fz", "Pz", "TP9", "TP10", "O9", "O10"]);

## 3.3 PSD (fmax=100) và đỉnh 50 Hz

In [ ]:
psd = raw.compute_psd(fmax=100, verbose="error")
psd.plot(average=False, amplitude=False);

# đỉnh 50 Hz: công suất trung bình kênh tại đỉnh so với nền 45–55 Hz (bỏ 49–51 Hz)
p, f = psd.get_data(return_freqs=True)
mean_db = 10 * np.log10(p.mean(axis=0))
band = (f >= 45) & (f <= 55)
peak_f = f[band][np.argmax(mean_db[band])]
floor = np.median(mean_db[band & ((f < 49) | (f > 51))])
print(f"Đỉnh trong 45–55 Hz: {peak_f:.2f} Hz, cao hơn nền {mean_db[f == peak_f][0] - floor:.1f} dB")

## 3.4 Kênh nghi hỏng (khách quan)
Tính trên **bản copy** (chỉ để chẩn đoán, không dùng tiếp), đã **average reference** — nếu giữ tham chiếu gốc FCz thì các kênh quanh FCz luôn có phương sai nhỏ giả tạo.
Robust z = (x − trung vị) / (1.4826·MAD) so với các kênh khác:
- **Phẳng**: độ lệch chuẩn (1–40 Hz) < 0.5 µV.
- **Phương sai quá lớn/nhỏ**: |z| của log std ở 1–40 Hz > 3.
- **Lớn ở 8–40 Hz**: z > 3 → không giải thích được bằng chớp mắt (chớp mắt tập trung < 8 Hz).
- **Nhiễu cao tần**: z của công suất 55–95 Hz > 3 (cơ, tiếp xúc kém).

Kênh trán chỉ lớn ở 1–40 Hz mà bình thường ở 8–40 Hz → do chớp mắt, để ICA xử lý, **không** coi là hỏng.

In [ ]:
def robust_z(x):
    return (x - np.median(x)) / (1.4826 * np.median(np.abs(x - np.median(x))))

def band_std(l_freq, h_freq):
    return raw.copy().filter(l_freq, h_freq, verbose="error").set_eeg_reference("average", verbose="error").get_data().std(axis=1)

std = band_std(1.0, 40.0)
p_hf, _ = raw.compute_psd(fmin=55, fmax=95, verbose="error").get_data(return_freqs=True)
hf_db = 10 * np.log10(p_hf.mean(axis=1))
ch_stats = pd.DataFrame({
    "std_uV": std * 1e6,
    "z_var_1_40": robust_z(np.log10(std)),
    "z_var_8_40": robust_z(np.log10(band_std(8.0, 40.0))),
    "z_hf": robust_z(hf_db),
    "hf_dB_vs_median": hf_db - np.median(hf_db),
}, index=raw.ch_names).round(2)

rules = {
    "phẳng": ch_stats["std_uV"] < 0.5,
    "phương sai quá lớn": ch_stats["z_var_1_40"] > 3,
    "phương sai quá nhỏ": ch_stats["z_var_1_40"] < -3,
    "lớn ở 8–40 Hz": ch_stats["z_var_8_40"] > 3,
    "nhiễu cao tần": ch_stats["z_hf"] > 3,
}
ch_stats["lý do"] = [", ".join(r for r, m in rules.items() if m[ch]) for ch in raw.ch_names]
blink_only = (ch_stats["lý do"] == "phương sai quá lớn") & ch_stats.index.str.startswith(("Fp", "AF"))
ch_stats.loc[blink_only, "lý do"] += " (chỉ < 8 Hz → chớp mắt, để ICA)"

print("Trung vị std (1–40 Hz, avg ref):", round(np.median(std) * 1e6, 2), "µV")
flagged = ch_stats[ch_stats["lý do"] != ""]
suspects = flagged[~blink_only[flagged.index]]
print(f"Kênh bị gắn cờ ({len(flagged)}):\n", flagged.sort_values("z_var_8_40", ascending=False), "\n")
print("=> NGHI HỎNG cần xem bằng mắt:", suspects.index.tolist() or "không có")
print("\n5 kênh nhiễu cao tần nhất:\n", ch_stats.nlargest(5, "z_hf"))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, col in zip(axes, ["z_var_1_40", "z_var_8_40", "z_hf"]):
    z = ch_stats[col].to_numpy()
    ax.bar(range(len(z)), z, color=np.where(np.abs(z) > 3, "tab:red", "tab:gray"))
    ax.axhline(3, ls="--", c="k"); ax.axhline(-3, ls="--", c="k")
    ax.set_ylabel(col)
axes[-1].set_xticks(range(len(raw.ch_names)), raw.ch_names, rotation=90, fontsize=6)
axes[0].set_title(f"sub-{SUB}: robust z từng kênh (avg ref)")
plt.tight_layout()

### 3.4b Cầu nối gel (bridged electrodes)
Hai điện cực bị gel nối thì ghi gần như cùng một tín hiệu → phương sai vẫn bình thường nên tiêu chí trên **không bắt được**. Dùng `mne.preprocessing.compute_bridged_electrodes` (khoảng cách điện giữa từng cặp kênh).

In [ ]:
bridged_idx, ed_matrix = mne.preprocessing.compute_bridged_electrodes(raw, verbose="error")
bridged_pairs = sorted((raw.ch_names[i], raw.ch_names[j]) for i, j in bridged_idx)
bridged_chs = sorted({ch for pair in bridged_pairs for ch in pair})
print(f"{len(bridged_pairs)} cặp cầu nối:", bridged_pairs)
print(f"{len(bridged_chs)} kênh liên quan:", bridged_chs)
mne.viz.plot_bridged_electrodes(raw.info, bridged_idx, ed_matrix,
                                title=f"sub-{SUB}: cầu nối gel", topomap_args=dict(vlim=(None, 5)));

## 3.5 `raw.plot` để xem bằng mắt
Vẽ bản 1–40 Hz, các kênh nghi hỏng tô đỏ (chỉ trên bản copy). Máy có `mne-qt-browser` nên ô này mở **cửa sổ Qt riêng** để cuộn/tương tác; phím `a` bật chú thích, click tên kênh để đánh dấu bad.

In [ ]:
raw_view = raw.copy().filter(1.0, 40.0, verbose="error")
raw_view.info["bads"] = sorted(set(suspects.index) | set(bridged_chs))  # tô đỏ: nghi hỏng + cầu nối
raw_view.plot(duration=20, n_channels=40, scalings=dict(eeg=50e-6), start=60);

# Giai đoạn 4: Làm sạch sub-01
**Đã chốt sau Giai đoạn 3:** bads = `["Fp2"]`; 21 kênh cầu nối xử lý bằng `interpolate_bridged_electrodes`; kênh mắt cho ICA = `Fp1`, `AFp2`; montage: TP9/TP10 đổi tên, O9/O10 mượn tọa độ I1/I2.

Thứ tự: lọc 0.1–40 Hz → nội suy cầu nối → ICA (Fp2 bị loại khỏi fit) → **chờ chốt thành phần** → apply ICA → nội suy Fp2 → average reference → resample.

## 4.1 Lọc, gán bads, nội suy cầu nối

In [ ]:
BADS = ["Fp2"]
EOG_CHS = ["Fp1", "AFp2"]  # Fp2 là bad nên dùng AFp2 (ngay cạnh, r = 0.97 với Fp2)
del raw_view  # giải phóng ~1.4 GB của bản xem ở 3.5

raw.filter(config.L_FREQ, config.H_FREQ, verbose="error")
raw.info["bads"] = BADS
# shortcut: hàm tạm bỏ info["bads"] nên Fp2 cũng góp vào spline nội suy cầu nối; Fp2 xa các cụm nên ảnh hưởng nhỏ
with mne.use_log_level("error"):
    mne.preprocessing.interpolate_bridged_electrodes(raw, bridged_idx)
assert raw.info["bads"] == BADS and len(raw.ch_names) == 127
print("bads:", raw.info["bads"], "| highpass/lowpass:", raw.info["highpass"], raw.info["lowpass"])

## 4.2 ICA
Fit trên bản copy lọc thông cao 1 Hz (trôi chậm làm ICA kém ổn định), sau đó áp lên bản 0.1 Hz. Kênh trong `info["bads"]` (Fp2) tự động bị loại khỏi fit.

In [ ]:
raw_ica = raw.copy().filter(l_freq=1.0, h_freq=None, verbose="error")
ica = mne.preprocessing.ICA(n_components=30, random_state=config.SEED, max_iter="auto")
ica.fit(raw_ica, decim=3, verbose="error")
print(ica)
print("Kênh dùng để fit:", len(ica.ch_names), "| Fp2 trong fit:", "Fp2" in ica.ch_names)
ica.plot_components(title=f"sub-{SUB}: 30 thành phần ICA");

## 4.3 Thành phần gợi ý là mắt (`find_bads_eog`)
- Chớp mắt / mắt dọc: tương quan với **Fp1**, **AFp2**.
- Cử động mắt ngang: dataset không có HEOG nên tạo kênh lưỡng cực ảo **F7 − F8** (chỉ để chấm điểm).

`find_bads_eog` lọc 1–10 Hz rồi tính tương quan từng nguồn với kênh mắt; gợi ý nguồn có |z| > 3. **Không tự loại** — chờ xác nhận.

In [ ]:
eog_idx, eog_scores = ica.find_bads_eog(raw_ica, ch_name=EOG_CHS, verbose="error")
raw_heog = mne.set_bipolar_reference(raw_ica, "F7", "F8", ch_name="HEOG", drop_refs=False, copy=True, verbose="error")
heog_idx, heog_scores = ica.find_bads_eog(raw_heog, ch_name="HEOG", verbose="error")
del raw_heog

scores = pd.DataFrame(np.vstack([*eog_scores, heog_scores]).T, columns=[*EOG_CHS, "HEOG"])
scores.index.name = "IC"
candidates = sorted({int(k) for k in [*eog_idx, *heog_idx]})
top = scores.abs().max(axis=1).nlargest(6).index  # 6 nguồn tương quan mạnh nhất, kể cả dưới ngưỡng
table = scores.loc[top].round(2)
table["var_%"] = [100 * ica.get_explained_variance_ratio(raw_ica, components=[k])["eeg"] for k in top]
table["gợi ý"] = ["CÓ" if k in candidates else "" for k in top]
print("Gợi ý từ Fp1/AFp2:", sorted(map(int, eog_idx)), "| từ HEOG (F7−F8):", sorted(map(int, heog_idx)))
print(table.round(2))
ica.plot_scores([*eog_scores, heog_scores], exclude=candidates, labels=[*EOG_CHS, "HEOG"]);

In [ ]:
# topomap + phổ + ảnh epoch của từng thành phần gợi ý; chuỗi thời gian 20 s (vẽ tĩnh để lưu được vào notebook)
ica.plot_properties(raw_ica, picks=candidates, verbose="error");
with mne.viz.use_browser_backend("matplotlib"):
    ica.plot_sources(raw_ica, picks=sorted(set(candidates) | set(top[:4])), start=300, stop=320, show_scrollbars=False);

## 4.4 Áp dụng thành phần đã chốt, nội suy Fp2
**Đã chốt:** loại IC0 (chớp mắt) và IC3 (cử động mắt ngang); giữ IC15 (khu trú gần F7, phổ phẳng → cơ/nhiễu kênh, không phải mắt).
ICA fit trên bản 1 Hz nhưng áp lên `raw` bản 0.1 Hz.

In [ ]:
ICA_EXCLUDE = {0: "chớp mắt (r = 0.98/0.99 với Fp1/AFp2)", 3: "cử động mắt ngang (r = 0.88 với F7−F8)"}
COMPARE_CHS = ["Fp1", "Fp2", "AFp2", "AF7", "AF8", "Fz", "Pz"]

raw_pre = raw.copy().pick(COMPARE_CHS)  # bản trước ICA (chỉ 7 kênh) để so sánh
del raw_ica  # giải phóng ~1.4 GB
ica.exclude = list(ICA_EXCLUDE)
ica.apply(raw, verbose="error")
raw.interpolate_bads(reset_bads=True, verbose="error")
assert raw.info["bads"] == []
print("Đã loại IC:", ica.exclude, "| bads còn lại:", raw.info["bads"])

## 4.5 So sánh trước/sau ở kênh trán
Cùng tham chiếu gốc (chưa average reference) để chỉ thấy tác động của ICA + nội suy Fp2. Pz là đối chứng: ICA mắt không được làm thay đổi nhiều ở vùng đỉnh.

In [ ]:
before = raw_pre.get_data() * 1e6
after = raw.get_data(picks=COMPARE_CHS) * 1e6
print(pd.DataFrame({"std trước (µV)": before.std(axis=1), "std sau (µV)": after.std(axis=1),
                    "sau/trước": after.std(axis=1) / before.std(axis=1)}, index=COMPARE_CHS).round(2))

t0, t1 = 300, 320  # đoạn có cả chớp mắt (306–311 s) và liếc ngang (313–314.5 s)
sl = slice(*raw.time_as_index([t0, t1]))
t = raw.times[sl]
fig, axes = plt.subplots(len(COMPARE_CHS), 1, figsize=(14, 10), sharex=True)
for ax, ch, b, a in zip(axes, COMPARE_CHS, before[:, sl], after[:, sl]):
    ax.plot(t, b, c="tab:red", lw=0.7, label="trước ICA")
    ax.plot(t, a, c="k", lw=0.7, label="sau ICA" + (" + nội suy" if ch in BADS else ""))
    ax.set_ylabel(f"{ch}\n(µV)")
    ax.legend(loc="upper right", fontsize=7)
axes[-1].set_xlabel("thời gian (s)")
axes[0].set_title(f"sub-{SUB}: trước/sau ICA (loại IC {ica.exclude}), 0.1–40 Hz, tham chiếu gốc")
plt.tight_layout()

## 4.6 Average reference và hạ mẫu

In [ ]:
del raw_pre
raw.set_eeg_reference("average", projection=False, verbose="error")
raw.resample(config.SFREQ_NEW, verbose="error")

assert raw.info["sfreq"] == config.SFREQ_NEW and len(raw.ch_names) == 127 and raw.info["bads"] == []
assert raw.info["custom_ref_applied"]  # average reference đã áp trực tiếp (không phải projector)
assert (raw.info["highpass"], raw.info["lowpass"]) == (config.L_FREQ, config.H_FREQ)
print(raw)
print("Độ lệch chuẩn trung vị:", round(np.median(raw.get_data().std(axis=1)) * 1e6, 2), "µV")

# bản ghi QC của sub-01 (Giai đoạn 6 sẽ lưu ra derivatives/v1/qc/sub-01_qc.json)
qc = {
    "subject": SUB,
    "bads": {"Fp2": "nhiễu 20–30 Hz liên tục, z = 5.5 ở 8–40 Hz"},
    "bridged_pairs": [list(p) for p in bridged_pairs],
    "ica_exclude": {str(k): v for k, v in ICA_EXCLUDE.items()},
    "eog_proxies": EOG_CHS,
}
qc

# Giai đoạn 5: Epoch, trial_id và kiểm tra ERP
## 5.1 Metadata: trial_id + nhãn lớp + đúng/sai
- `trial_id`, `condition`, `event_code`, `source_row` lấy từ `read_stimulus_metadata` của TV1 → cùng cách đánh số với v0 và `folds.json`.
- Thêm `n_resp`, `rt`, `correct` theo quy tắc đã chốt: mỗi phản hồi gắn vào kích thích gần nhất trước nó; target đúng = có bấm, standard/distractor đúng = không bấm; không ngưỡng RT.

In [ ]:
from create_v0_data import read_stimulus_metadata  # hàm của TV1, file ở thư mục gốc repo

events_path = bids_path.copy().update(suffix="events", extension=".tsv").fpath
metadata = read_stimulus_metadata(events_path)  # đã kiểm tra đủ 660 kích thích, onset tăng dần

ev_all = pd.read_csv(events_path, sep="\t")
codes = ev_all["event_type"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
resp_on = np.sort(ev_all.loc[codes == "S 1", "onset"].to_numpy(float))
stim_on = metadata["onset"].to_numpy(float)
idx = np.searchsorted(stim_on, resp_on, side="right") - 1  # kích thích gần nhất trước mỗi phản hồi
assert (idx >= 0).all(), "có phản hồi trước kích thích đầu tiên"
rt = pd.Series(resp_on - stim_on[idx], index=idx)
metadata["n_resp"] = rt.groupby(level=0).size().reindex(metadata.index, fill_value=0).to_numpy()
metadata["rt"] = rt.groupby(level=0).min().reindex(metadata.index).to_numpy()  # RT phản hồi đầu tiên
metadata["correct"] = (metadata["condition"] == "target") == (metadata["n_resp"] > 0)

print(metadata.columns.tolist())
print(pd.crosstab(metadata["condition"], metadata["correct"], margins=True))
metadata.head()

## 5.2 Events và cắt epoch (reject 150 µV)
Onset (giây) → chỉ số mẫu ở 250 Hz như v0. Đối chiếu với annotation `stimulus` mà `read_raw_bids` đọc từ `.vmrk` để chắc thời điểm khớp.

In [ ]:
ann = raw.annotations
ann_on = ann.onset[ann.description == "stimulus"] - (raw.first_samp / raw.info["sfreq"] if ann.orig_time else 0)
print("Lệch onset events.tsv vs annotation (max, ms):", round(np.abs(ann_on - stim_on).max() * 1e3, 3))

event_samples = raw.time_as_index(stim_on, use_rounding=True) + raw.first_samp
events = np.column_stack([event_samples, np.zeros(len(metadata), int), metadata["event_code"].to_numpy(int)])
assert len(metadata) == len(events) and (np.diff(event_samples) > 0).all()

epochs_all = mne.Epochs(raw, events, event_id=config.EVENT_ID, tmin=config.TMIN, tmax=config.TMAX,
                        baseline=(config.TMIN, 0), reject=dict(eeg=150e-6), metadata=metadata,
                        preload=True, reject_by_annotation=False, event_repeated="error", verbose="error")

# trial_id sau reject: tập con của ban đầu, không trùng, khớp nhãn events
kept = epochs_all.metadata["trial_id"].to_numpy()
assert set(kept) <= set(metadata["trial_id"]) and len(np.unique(kept)) == len(kept)
assert np.array_equal(epochs_all.events[:, 2], epochs_all.metadata["event_code"].to_numpy())
print(f"Còn {len(epochs_all)}/{len(metadata)} epoch sau reject ({100 * (1 - len(epochs_all) / len(metadata)):.1f}% bị loại)")

## 5.3 Drop log, chỉ giữ thử nghiệm đúng, số epoch mỗi lớp

In [ ]:
drop_logs = [log for log in epochs_all.drop_log if log]
print("Kênh gây loại epoch (số epoch có kênh đó vượt ngưỡng):")
print(pd.Series([ch for log in drop_logs for ch in log]).value_counts().head(10))
print("Epoch bị loại CHỈ vì Fp1:", sum(log == ("Fp1",) for log in drop_logs), "/", len(drop_logs))

epochs = epochs_all[np.flatnonzero(epochs_all.metadata["correct"].to_numpy())]
assert epochs.metadata["correct"].all()

summary = pd.DataFrame({
    "tổng": metadata["condition"].value_counts(),
    "làm đúng": metadata.loc[metadata["correct"], "condition"].value_counts(),
    "còn sau reject": epochs_all.metadata["condition"].value_counts(),
    "CUỐI (đúng + sạch)": epochs.metadata["condition"].value_counts(),
}).reindex(list(config.EVENT_ID))
summary["% đúng bị reject"] = (100 * (1 - summary["CUỐI (đúng + sạch)"] / summary["làm đúng"])).round(1)
print(summary)
epochs

## 5.4 ERP ở Pz và Fz; `plot_joint` của target
Kỳ vọng: **P3b** dương ở Pz ~300–500 ms cho target; **P3a** dương phía trán–trung tâm (Fz), sớm hơn, cho distractor.

In [ ]:
evokeds = {c: epochs[c].average() for c in config.EVENT_ID}
for ch in ["Pz", "Fz"]:
    mne.viz.plot_compare_evokeds(evokeds, picks=ch, title=f"sub-{SUB}: ERP tại {ch}",
                                 colors=dict(standard="tab:gray", target="tab:red", distractor="tab:blue"))

# giá trị lớn nhất 250–600 ms (có thể âm nếu không có sóng dương) và trung bình 300–500 ms
rows = []
for c, ev in evokeds.items():
    for ch in ["Pz", "Fz"]:
        win = ev.copy().pick(ch).crop(0.25, 0.6)
        i = win.data[0].argmax()
        rows.append({"lớp": c, "kênh": ch, "n": ev.nave, "đỉnh (ms)": round(win.times[i] * 1e3),
                     "biên độ đỉnh (µV)": round(win.data[0, i] * 1e6, 2),
                     "TB 300–500 ms (µV)": round(ev.copy().pick(ch).crop(0.3, 0.5).data.mean() * 1e6, 2)})
print(pd.DataFrame(rows).set_index(["kênh", "lớp"]).sort_index())

evokeds["target"].plot_joint(times=[0.3, 0.4, 0.5], title=f"sub-{SUB}: target");